[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/kirchhoff/kirchhoff.ipynb)

# Kirchhoff's Equations

This example follows a solid body moving through still, ideal fluid, one without viscosity that cannot be compressed, as described by Kirchhoff's equations. A body moving through water has to push the water aside, and the water it sets moving carries kinetic energy of its own. That energy depends only on how the body moves, quadratically, so the fluid acts as extra inertia carried along with the body: its added mass. Unlike the body's own mass, the added mass depends on the direction of motion. An elongated body moving end-on pushes little water aside, moving broadside a lot, so its momentum no longer points along its motion, and a body launched nearly end-on swings round broadside: the Munk moment that makes airships and submarines unstable without fins.

Everything here is a quadratic form. The body, an ellipsoid, is modelled by its shape, a map on directions holding its squared semi-axes along its own axes, and the ellipsoids confocal with it, every squared semi-axis grown by the same amount, are the pencil `shape + widening * Direction`. The flow around the body is built from that pencil: weighted by the body's volume over each confocal ellipsoid's, the inverses of the pencil average to the fluid's response to a shift, and their outermorphisms, which act on couples, to its response to a turn. Added to the body's own, the result is one map from rates to momenta, the inertia, whose form `rate & inertia(rate)` is twice the kinetic energy of body and fluid together. Left alone, body and fluid keep their total momentum, the impulse, fixed in space, and the body moves at the rate whose momentum is the impulse as the body sees it, `mobility(motor << impulse)`.

In the vector notation of fluid dynamics, with the body's velocity $\mathbf U$, angular velocity $\boldsymbol\omega$, impulse $\mathbf P$ and angular impulse $\boldsymbol\Pi$ in its own frame, Kirchhoff's equations read as $\dot{\mathbf P} = \mathbf P \times \boldsymbol\omega$ and $\dot{\boldsymbol\Pi} = \boldsymbol\Pi \times \boldsymbol\omega + \mathbf P \times \mathbf U$, and Lamb's coefficients of the ellipsoid as $\alpha_0 = abc \int_0^\infty \frac{d\lambda}{(a^2 + \lambda)\Delta}$, $\Delta = \sqrt{(a^2+\lambda)(b^2+\lambda)(c^2+\lambda)}$, with the added mass $\frac{\alpha_0}{2 - \alpha_0}$ times the displaced fluid's along the first axis.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D
from examples.animation import save_figure, show_animation
from examples.fluids.kirchhoff import render
from examples.fluids.kirchhoff.core import quadrature

ga = PGA3D
mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
# Offsets, and the velocities of points, are points at infinity: antivectors with intrinsic orientation.
Direction = Point.derive.containing("w")
Twist = ga.gatype.bivector()
Forque = ga.gatype.antibivector()
# Couples are forques along a line at infinity: the momentum of a turn.
Couple = Forque.derive.containing("w")
Motor = ga.gatype.rotor()
Stretch = ga.gatype((Direction, Direction))                # Direction <- Direction: the unit sphere onto the body
Shape = ga.gatype((Direction, Direction))                  # Direction <- Direction: squared semi-axes along the body's axes
Inertia = ga.gatype((Forque, Twist))                       # Forque <- Twist: momentum per rate

# The body's centre, the planes through it square to its axes, and the directions along those axes.
centre = mv.point([0, 0, 0])                               # [] Point
axes = stack([mv.x, mv.y, mv.z])                           # [3] Plane
x, y, z = axes
directions = axes.dual()                                   # [3] Direction
# The velocity of the centre under a rate, and the couple of a turn: half the sum, over the axes, of
# each direction joined with its own velocity. A shift moves no direction, so it has no couple.
centre_velocity = centre.commutator(Twist)                 # Direction <- Twist
turn_couple = (directions & directions.commutator(Twist)).sum(axis=-1) / 2   # Couple <- Twist

## 1. The fluid's inertia

A shift's momentum is a line through the body's centre, the centre joined with its mass times its velocity, and a turn's is a couple. The solid's mass is the same whichever way it moves, and its inertia under a turn is a fifth of its mass times the sum of its two squared semi-axes in the plane of the turn. The fluid's inertia comes from the confocal pencil, in units of the mass of the fluid the body displaces. Under a turn it carries the square of the difference of those two squared semi-axes, the discriminant `spread(spread) - 4 * areas`, so a turn about an axis of symmetry moves no fluid. Each spheroid, from a flat disc to a long rod, is compared with its own solid at the same density.

In [ ]:
# Spheroids from a disc to a rod: the semi-axis along the axis of symmetry over the other two. The samples
# of the confocal family.
aspects = np.geomspace(0.1, 10, 41)
n_widenings = 48

In [ ]:
def stretch(semi_axes: np.ndarray) -> Stretch:
    """Ellipsoids in their own frames: the unit sphere stretched by each semi-axis `[..., 3]` along its axis."""
    # Each direction carried along each axis by its part along it, times the semi-axis.
    return (directions * semi_axes * (axes & Direction)).sum(axis=-1)          # [...] Stretch


def inertia(shape: Shape, density: np.ndarray, fluid_density: np.ndarray) -> Inertia:
    """The inertia of a solid ellipsoid of each shape and the given density, together with the ideal
    fluid of the given density it moves through."""
    widenings, weights = quadrature(n_widenings)
    # The mean squared semi-axis sets the scale of the widenings, and the body's volume is a square root of
    # the shape's determinant.
    scale = shape.trace() / 3                                                    # [...] Scalar
    volume = 4 / 3 * np.pi * shape.det().square_root()                           # [...] Scalar
    # The confocal ellipsoids, every squared semi-axis grown by the same widening, each weighed by the
    # body's volume over its own.
    confocal = shape[..., None] + (scale[..., None] * widenings) * Direction     # [..., widenings] Shape
    weight = (shape.det()[..., None] / confocal.det()).square_root() * scale[..., None] * weights   # [..., widenings] Scalar
    # Averaged over the confocal family, the inverse shape gives the response to a shift along each axis,
    # and its outermorphism the response to a turn in each plane. The first has trace two.
    shifting = (confocal.inverse() * weight).sum(axis=-1)                        # [...] Direction <- Direction
    turning = (confocal.inverse().outermorphism(Couple) * weight).sum(axis=-1)   # [...] Couple <- Couple
    # On each couple, the outermorphism multiplies the squared semi-axes in its plane and `spread` adds them:
    # the shape's trace less the squared semi-axis square to the plane. Their discriminant is the square of
    # their difference, which vanishes for a turn about an axis of symmetry.
    areas = shape.outermorphism(Couple)                                          # [...] Couple <- Couple
    spread = shape.trace() * Couple - shape.det() * areas.inverse()              # [...] Couple <- Couple
    discriminant = spread(spread) - 4 * areas                                    # [...] Couple <- Couple
    # The fluid's mass under a shift and its inertia under a turn, per unit of the displaced fluid's mass.
    shift_mass = (2 * Direction - shifting).solve(shifting)                      # [...] Direction <- Direction
    turn_mass = (2 * Couple - spread(turning)).solve(discriminant(turning)) / 5  # [...] Couple <- Couple
    # The solid's own: its mass whichever way it moves, and a fifth of it times the spread under a turn.
    masses = (volume * density) * Direction + (volume * fluid_density) * shift_mass        # [...] Direction <- Direction
    turns = (volume * density / 5) * spread + (volume * fluid_density) * turn_mass         # [...] Couple <- Couple
    return (centre & masses(centre_velocity)) + turns(turn_couple)               # [...] Forque <- Twist


semi_axes = np.stack([aspects, np.ones_like(aspects), np.ones_like(aspects)], axis=-1)   # [aspects, 3]
stretches = stretch(semi_axes)                                                 # [aspects] Stretch
shapes = stretches(stretches)                                                  # [aspects] Shape
# The solid in a vacuum, and the fluid around a body without mass, at the same density.
solid = inertia(shapes, 1, 0)[:, None]                                         # [aspects, 1] Forque <- Twist
fluid = inertia(shapes, 0, 1)[:, None]                                         # [aspects, 1] Forque <- Twist
# A shift along the axis of symmetry, a shift across it, and a turn about an axis across it.
rates = stack([mv.w ^ x, mv.w ^ y, z ^ x])                                     # [3] Twist
ratios = (rates & fluid(rates)) / (rates & solid(rates))                       # [aspects, 3] Scalar
save_figure(render.draw_coefficients(aspects, ratios), "kirchhoff_added_mass");

The curves are the fluid's kinetic energy over the solid's: blue for a shift along the axis of symmetry, orange for a shift across it, green for a turn about an axis across it. A sphere, at the grey line, carries half the mass of the fluid it displaces whichever way it moves, and turns without moving any. A disc moving face-on, or turning about a diameter, carries several times its own volume of fluid along; a rod moving broadside carries about its own volume, and moving end-on almost none.

## 2. Turning broadside

A body given a blow and then left to coast keeps the impulse of the blow, body and fluid together, fixed in space. Each step moves the body at the rate whose momentum is the impulse as the body sees it, taken halfway through the step, and all cases step together as one batch. Without the fluid, the body glides straight along the blow. In the fluid, a body at a slant needs less momentum to move along its length than across it, so it does not move along its impulse: its centre drifts off the impulse's line, which then passes beside the centre and turns the body. Launched nearly end-on, it swings round through broadside to nearly end-on the other way and back, like a pendulum released near the top; launched nearly broadside, it rocks about broadside.

In [ ]:
# The gliding body: its semi-axes and density, the fluid's density in each case, without the fluid, nearly
# end-on and nearly broadside, the blow along the first axis of space, and the long axis's angle from the
# blow at the start.
body_semi_axes = np.array([3, 1, 1])
density = 1
fluid_densities = np.array([0, 1, 1])
blow = 12
headings = np.array([0.1, 0.1, np.pi / 2 - 0.1])
# The time step, the steps, the steps per frame, the samples around the body's outline, and the duration
# of a frame.
dt = 0.2
n_steps = 200
n_steps_per_frame = 2
n_outline = 48
duration_ms = 50

Each lane is one case, from the top: the body without the fluid, then in the fluid launched nearly end-on and nearly broadside, all seen from above. The dashed line is the impulse, the solid line the path of the centre, and the dot the nose, one end of the long axis. This cell takes about five seconds, most of it drawing the frames.

In [ ]:
def glide(motors: Motor, impulse: Forque, inertias: Inertia, dt: float, n_steps: int):
    """The bodies' placements from the start and after each step, with no force on body or fluid: each
    moves at the rate whose momentum is the impulse as it sees it, taken at the midpoint of the step."""
    mobility = inertias.inverse()                                              # [cases] Twist <- Forque
    yield motors
    for _ in range(n_steps):
        halfway = motors * (mobility(motors << impulse) * (-dt / 4)).exp()
        motors = motors * (mobility(halfway << impulse) * (-dt / 2)).exp()
        yield motors


def frames(motors: Motor):
    """Per frame, every case's outline, centre and nose."""
    angles = np.linspace(0, 2 * np.pi, n_outline + 1)
    # The outline in the plane of the first two axes: the unit circle turned out by rotors, stretched onto the body.
    outline = centre + body(((x ^ y) * (-angles / 2)).exp() >> long_axis)    # [outline + 1] Point
    nose = centre + body(long_axis)                                            # [] Point
    for index in range(0, motors.shape[0], n_steps_per_frame):
        placed = motors[index]                                                 # [cases] Motor
        yield placed[:, None] >> outline, placed >> centre, placed >> nose


body = stretch(body_semi_axes)                                                 # [] Stretch
inertias = inertia(body(body), density, fluid_densities)                       # [cases] Forque <- Twist
long_axis = directions[0]                                                      # [] Direction
impulse = blow * (centre & long_axis)                                          # [] Forque
starts = ((x ^ y) * (-headings / 2)).exp()                                     # [cases] Motor
motors = stack(list(glide(starts, impulse, inertias, dt, n_steps)))            # [steps + 1, cases] Motor
show_animation(render.animate_gliding(impulse, frames(motors)), "kirchhoff_gliding", duration_ms)

In [ ]:
# checks
# A sphere carries half the mass of the fluid it displaces whichever way it moves, and turns without
# moving any.
ball = stretch(np.ones(3))                                                     # [] Stretch
sphere = inertia(ball(ball), 0, 1)                                             # [] Forque <- Twist
np.testing.assert_allclose((rates & sphere(rates)).to_array(), [2 * np.pi / 3, 2 * np.pi / 3, 0], atol=1e-12)
# The kinetic energy of body and fluid is kept to within the step's error.
seen = motors << impulse                                                       # [steps + 1, cases] Forque
energies = (seen & inertias.inverse()(seen)).to_array()                        # [steps + 1, cases]
np.testing.assert_allclose(energies / energies[0], 1, rtol=1e-2)
# Without the fluid the body keeps its heading; launched nearly broadside it stays so; nearly end-on it
# turns past broadside.
pointing = motors >> long_axis                                                 # [steps + 1, cases] Direction
angles = np.arctan2((y & pointing).to_array(), (x & pointing).to_array())      # [steps + 1, cases]
np.testing.assert_allclose(angles[:, 0], headings[0], atol=1e-12)
assert np.abs(angles[:, 2] - np.pi / 2).max() < 2 * (np.pi / 2 - headings[2])
assert angles[:, 1].max() > np.pi / 2